# Advanced Unsupervised Clustering and Data Quality Monitoring (Julia)

**Project Title:** Advanced Unsupervised Clustering and Data Quality Monitoring

**Objective:** Develop a robust Julia script to perform extensive comparative analysis on material master data using optimized, advanced unsupervised learning models. This implementation leverages Julia's high-performance Just-In-Time (JIT) compilation for speed.

**Deliverables:**
1.  **Code File:** A single, complete Julia notebook.
2.  **Output File:** A single text file named `analysis_summary.txt` containing all results.

In [ ]:
import Pkg
Pkg.add(["CSV", "DataFrames", "Clustering", "MultivariateStats", "Flux", "Statistics", "LinearAlgebra", "Random", "Distances", "JLD2", "FileIO", "DecisionTree", "Plots"])

using CSV, DataFrames, Clustering, MultivariateStats, Flux, Statistics, LinearAlgebra, Random, Distances, JLD2, FileIO, DecisionTree, Plots

In [ ]:
# Configuration
const DATA_FILE = joinpath("data", "MARC all.csv")
const OUTPUT_FILE = "analysis_summary.txt"
const MODEL_DIR = "models_julia"

if !isdir(MODEL_DIR)
    mkdir(MODEL_DIR)
end

In [ ]:
# Checkpointing Helper Function
function load_or_compute(filepath, compute_func; force=false)
    if isfile(filepath) && !force
        println("Loading cached results from $filepath...")
        d = load(filepath)
        return d["data"]
    else
        println("Computing results...")
        result = compute_func()
        println("Saving results to $filepath...")
        save(filepath, "data", result)
        return result
    end
end


function load_and_preprocess_data(filepath)
    println("Loading data from $filepath...")
    df = CSV.read(filepath, DataFrame)
    println("Data loaded: $(size(df))")

    # 1. Drop Identifiers and Low Variance
    # Explicitly drop MAT_ID if it exists as it is a unique identifier
    if "MAT_ID" in names(df)
        select!(df, Not("MAT_ID"))
        println("Dropped MAT_ID column.")
    end

    cols_to_drop = Symbol[]
    for col in names(df)
        if length(unique(df[!, col])) <= 1
            push!(cols_to_drop, Symbol(col))
        end
    end
    select!(df, Not(cols_to_drop))
    println("Dropped $(length(cols_to_drop)) low-variance columns.")

    # 2. Imputation
    for col in names(df)
        if eltype(df[!, col]) <: Number
            m = mean(skipmissing(df[!, col]))
            df[!, col] = coalesce.(df[!, col], m)
        else
            non_missing = collect(skipmissing(df[!, col]))
            if isempty(non_missing)
                fill_val = "Unknown"
            else
                counts = Dict{Any, Int}()
                for v in non_missing
                    counts[v] = get(counts, v, 0) + 1
                end
                fill_val = argmax(counts)
            end
            df[!, col] = coalesce.(df[!, col], fill_val)
        end
    end

    # 3. Encoding
    println("Encoding categorical features...")
    cat_cols = names(df, AbstractString)
    df_encoded = select(df, names(df, Number))
    
    MAX_CARDINALITY = 50
    
    for col in cat_cols
        vals = unique(df[!, col])
        n_unique = length(vals)
        
        if n_unique > MAX_CARDINALITY
            println("Skipping encoding for column '$col' due to high cardinality ($n_unique > $MAX_CARDINALITY)")
            continue
        end
        
        if n_unique > 1
            for val in vals[2:end]
                new_col_name = "$(col)_$(val)"
                df_encoded[!, new_col_name] = (df[!, col] .== val)
            end
        end
    end

    println("Encoded DataFrame size: $(size(df_encoded))")

    # Convert to Matrix{Float32}
    # Use a more memory efficient way if possible, but with reduced columns it should be fine
    data_matrix = Matrix{Float32}(df_encoded)
    data_matrix = permutedims(data_matrix)

    # 4. Scaling
    println("Scaling data...")
    means = mean(data_matrix, dims=2)
    stds = std(data_matrix, dims=2)
    stds[stds .== 0] .= 1.0f0
    data_scaled = (data_matrix .- means) ./ stds

    # 5. PCA
    println("Running PCA...")
    M = fit(PCA, data_scaled; pratio=0.95)
    data_pca = MultivariateStats.transform(M, data_scaled)

    println("Data shape after PCA: $(size(data_pca))")
    return data_pca, df_encoded, M, (means, stds)
end


## Data Preprocessing & Feature Engineering
- Load `master_data.csv`
- Drop low-variance features
- Impute missing values
- Encode categorical features
- Scale data
- Perform PCA

In [ ]:
# Execute Loading with Checkpointing
data_pca, df_encoded, pca_model, scaler_params = load_or_compute(joinpath(MODEL_DIR, "data_preprocessed.jld2"), 
    () -> load_and_preprocess_data(DATA_FILE)
)

## Model 1: K-Means Optimization
- Test range of K (5-20)
- Use Silhouette Score to find optimal K

In [ ]:
function calculate_silhouette_score(assignments, counts, data)
    # Calculate Silhouette Score using full data
    # We use the signature silhouettes(assignments, counts, X) if available
    # to avoid precomputing the huge distance matrix.
    
    # Ensure data is matrix
    # silhouettes in Clustering.jl (some versions) supports X directly with counts
    try
        sils = silhouettes(assignments, counts, data)
        return mean(sils)
    catch e
        println("Warning: silhouettes(assignments, counts, data) failed: $e")
        println("Falling back to subsampled distance matrix due to memory constraints for O(N^2) operation.")
        # Fallback if the direct method fails (e.g. older version)
        # We MUST subsample here otherwise we crash RAM.
        # But user asked for full data. We will try a larger sample or warn.
        sample_size = 20000
        n_samples = size(data, 2)
        indices = randperm(n_samples)[1:min(n_samples, sample_size)]
        sub_data = data[:, indices]
        sub_assignments = assignments[indices]
        P = pairwise(Euclidean(), sub_data, dims=2)
        sils = silhouettes(sub_assignments, P)
        return mean(sils)
    end
end

function train_kmeans_julia(data, k_range)
    println("Training K-Means...")
    best_score = -1.0
    best_k = -1
    best_model = nothing
    
    results = Dict()

    for k in k_range
        # Clustering.jl kmeans expects Features x Samples
        result = kmeans(data, k; maxiter=100, display=:none)
        
        # Calculate Silhouette
        score = calculate_silhouette_score(result.assignments, result.counts, data)
        
        results[k] = score
        println("K=$k, Silhouette=$(round(score, digits=4))")
        
        if score > best_score
            best_score = score
            best_k = k
            best_model = result
        end
    end
            
    return best_model, best_k, best_score, results
end

if data_pca !== nothing
    kmeans_results = load_or_compute(joinpath(MODEL_DIR, "kmeans_results_k2_30.jld2"), 
        () -> train_kmeans_julia(data_pca, 2:30)
    )
    kmeans_model, best_k_kmeans, best_score_kmeans, _ = kmeans_results
end

In [ ]:

if isdefined(Main, :kmeans_model) && kmeans_model !== nothing
    println("Plotting K-Means Results...")

    # 1. Silhouette vs K
    _, _, _, k_results_dict = kmeans_results

    ks     = sort(collect(keys(k_results_dict)))
    scores = [k_results_dict[k] for k in ks]

    # Base silhouette plot
    p_sil = plot(
        ks, scores,
        title  = "Silhouette Score vs K",
        xlabel = "K", ylabel = "Silhouette Score",
        marker = :o, legend = false,
        xticks = (ks, string.(ks))
    )

    # Max-K (best score) lines + annotation
    max_idx = argmax(scores)
    k_max   = ks[max_idx]
    s_max   = scores[max_idx]

    vline!(p_sil, [k_max], color = :red,  lw = 1.5, label = false)
    hline!(p_sil, [s_max], color = :red, lw = 1.5, label = false)
    
    annotate!(
        p_sil,
        (k_max, s_max,
         Plots.text("(k=$k_max, s=$(round(s_max; digits=2)))", 9, :black, :left))
    )

    # K = 7 lines and annotation
    if 7 in ks
        k7_idx = findfirst(==(7), ks)
        s7     = scores[k7_idx]

        vline!(p_sil, [7],   color = :blue, lw = 1.5, label = false)
        hline!(p_sil, [s7],  color = :blue, lw = 1.5, label = false)

        annotate!(
            p_sil,
            (7, s7,
             Plots.text("(k=7, s=$(round(s7; digits=2)))", 9, :black, :left))
        )
    end

    # K = 9 lines and annotation
    if 9 in ks
        k9_idx = findfirst(==(9), ks)
        s9     = scores[k9_idx]

        vline!(p_sil, [9],    color = :orange, lw = 1.5, label = false)
        hline!(p_sil, [s9],   color = :orange, lw = 1.5, label = false)

        annotate!(
            p_sil,
            (9, s9,
             Plots.text("(k=9, s=$(round(s9; digits=2)))", 9, :black, :left))
        )
    end

    # K = 22 lines and annotation
    if 22 in ks
        k22_idx = findfirst(==(22), ks)
        s22     = scores[k22_idx]

        vline!(p_sil, [22],   color = :khaki, lw = 1.5, label = false)
        hline!(p_sil, [s22],  color = :khaki, lw = 1.5, label = false)

        annotate!(
            p_sil,
            (22, s22,
             Plots.text("(k=22, s=$(round(s22; digits=2)))", 9, :black, :left))
        )
    end

    display(p_sil)

    # 2. Cluster Scatter - Best K (current model)
    n_plot  = size(data_pca, 2)
    indices = 1:n_plot

    p_scatter_best = scatter(
        data_pca[1, indices], data_pca[2, indices],
        group  = kmeans_model.assignments[indices],
        title  = "K-Means Clusters (K=$(best_k_kmeans))",
        xlabel = "PC1", ylabel = "PC2",
        legend = :outertopright,
        markersize = 2, markerstrokewidth = 0, alpha = 0.6
    )
    display(p_scatter_best)

    # 3. Cluster Scatter - K=7 (recompute if needed)
    if 7 in ks
        println("Computing K=7 clusters...")
        
        # Re-run kmeans for K=7 using same data
        k7_result = kmeans(data_pca[1:2, :], 7; maxiter=200)  # Use PC1,PC2 only
        k7_assignments = k7_result.assignments
        
        p_scatter7 = scatter(
            data_pca[1, indices], data_pca[2, indices],
            group  = k7_assignments[indices],
            title  = "K-Means Clusters (K=7)",
            xlabel = "PC1", ylabel = "PC2",
            legend = :outertopright,
            markersize = 2, markerstrokewidth = 0, alpha = 0.6
        )
        display(p_scatter7)
    end
end


## Model 2: X-Means (Simulated)
- Simulate X-Means using BIC score to select optimal K

In [ ]:
function train_xmeans_simulated(data, max_k=20)
    println("Training X-Means (Simulated via BIC)...")
    best_bic = Inf
    best_k = -1
    best_model = nothing
    
    d, n_samples = size(data)
    
    for k in 2:max_k
        result = kmeans(data, k; maxiter=100, display=:none)
        
        # Calculate BIC
        # WCSS (Within-Cluster Sum of Squares) is result.totalcost
        wcss = result.totalcost
        
        # Variance estimate
        # variance = wcss / (n_samples - k)
        # This is a simplified BIC for K-Means
        # BIC = n * ln(wcss/n) + k * ln(n) * d? 
        # Standard BIC: n * ln(RSS/n) + k * ln(n)
        
        bic = n_samples * log(wcss / n_samples) + k * log(n_samples)
        
        if bic < best_bic
            best_bic = bic
            best_k = k
            best_model = result
        end
    end
            
    return best_model, best_k
end

if data_pca !== nothing
    xmeans_results = load_or_compute(joinpath(MODEL_DIR, "xmeans_results.jld2"), 
        () -> train_xmeans_simulated(data_pca)
    )
    xmeans_model, best_k_xmeans = xmeans_results
end

In [ ]:
if isdefined(Main, :xmeans_model) && xmeans_model !== nothing
    println("Plotting X-Means Results...")
    
    n_plot = size(data_pca, 2) # Use all data
    indices = 1:n_plot
    
    p_xm = scatter(data_pca[1, indices], data_pca[2, indices], 
        group=xmeans_model.assignments[indices], 
        title="X-Means Clusters (K=$(best_k_xmeans))",
        xlabel="PC1", ylabel="PC2",
        legend=:outertopright, markersize=2, markerstrokewidth=0, alpha=0.6
    )
    display(p_xm)
#     savefig(p_xm, joinpath(MODEL_DIR, "xmeans_scatter.png"))
end

## Model 3: Autoencoder + K-Means
- Train a Shallow Autoencoder (Flux.jl)
- Cluster on the Latent Space

In [ ]:
function train_autoencoder_clustering(data, encoding_dim=10, epochs=20)
    println("Training Autoencoder...")
    input_dim = size(data, 1)
    n_samples = size(data, 2)
    
    # Define Model
    encoder = Chain(
        Dense(input_dim, 128, relu),
        Dense(128, 64, relu),
        Dense(64, encoding_dim, relu)
    )
    
    decoder = Chain(
        Dense(encoding_dim, 64, relu),
        Dense(64, 128, relu),
        Dense(128, input_dim) # Linear output for scaled data
    )
    
    model = Chain(encoder, decoder)
    
    # Loss function
    # Loss function (explicit model arg)
    loss(m, x) = Flux.mse(m(x), x)
    
    # Optimizer
    opt = Adam(0.001)
    opt_state = Flux.setup(opt, model)
    
    # Data Loader
    # Flux expects batches. 
    batch_size = 256
    data_loader = Flux.DataLoader(data, batchsize=batch_size, shuffle=true)
    
    # Training Loop
    for epoch in 1:epochs
        for x in data_loader
            val, grads = Flux.withgradient(model) do m
                loss(m, x)
            end
            Flux.update!(opt_state, model, grads[1])
        end
        # Calculate epoch loss
        current_loss = loss(model, data)
        println("Epoch $epoch/$epochs, Loss: $(round(current_loss, digits=4))")
    end
    
    # Extract Latent
    latent_data = encoder(data)
    
    println("Clustering on Latent Space...")
    kmeans_ae = kmeans(latent_data, 5; maxiter=100) # Fixed K=5 or search
    
    return model, kmeans_ae, latent_data
end

if data_pca !== nothing
    ae_results = load_or_compute(joinpath(MODEL_DIR, "ae_results.jld2"), 
        () -> train_autoencoder_clustering(data_pca)
    )
    ae_model, ae_kmeans, latent_data = ae_results
    
    # Save individual state for consistency if needed, though ae_results has it
    model_state = Flux.state(ae_model)
    save(joinpath(MODEL_DIR, "autoencoder_state.jld2"), "state", model_state)
end

In [ ]:
if isdefined(Main, :ae_kmeans) && ae_kmeans !== nothing
    println("Plotting Autoencoder Results...")
    
    # Plotting on the original PCA space for consistency, colored by AE clusters
    n_plot = size(data_pca, 2) # Use all data
    indices = 1:n_plot
    
    p_ae = scatter(data_pca[1, indices], data_pca[2, indices], 
        group=ae_kmeans.assignments[indices], 
        title="Autoencoder Clusters (Latent K=$(size(ae_kmeans.centers, 2)))",
        xlabel="PC1", ylabel="PC2",
        legend=:outertopright, markersize=2, markerstrokewidth=0, alpha=0.6
    )
    display(p_ae)
#     savefig(p_ae, joinpath(MODEL_DIR, "autoencoder_scatter.png"))
end

## Result Aggregation & Analysis
- Calculate Outliers
- Calculate Davies-Bouldin Index
- Identify Top Features
- Generate Summary Report

In [ ]:
function get_top_features(feature_names, centers, top_n=3)
    # Calculate variance of each feature across cluster centers
    # centers is Features x K
    # We want features that vary the most between clusters
    
    feat_vars = var(centers, dims=2)
    feat_vars = vec(feat_vars)
    
    indices = sortperm(feat_vars, rev=true)[1:min(top_n, length(feat_vars))]
    return feature_names[indices]
end

# Davies-Bouldin implementation (simplified or manual if package missing)
function davies_bouldin(data, assignments, centers)
    k = size(centers, 2)
    n_features, n_samples = size(data)
    
    # 1. Calculate intra-cluster dispersion (average distance to centroid)
    dispersions = zeros(k)
    counts = zeros(Int, k)
    
    for i in 1:n_samples
        c = assignments[i]
        dispersions[c] += euclidean(data[:, i], centers[:, c])
        counts[c] += 1
    end
    dispersions ./= counts
    
    # 2. Calculate DB Index
    db_sum = 0.0
    for i in 1:k
        max_ratio = 0.0
        for j in 1:k
            if i != j
                dist_centers = euclidean(centers[:, i], centers[:, j])
                ratio = (dispersions[i] + dispersions[j]) / dist_centers
                if ratio > max_ratio
                    max_ratio = ratio
                end
            end
        end
        db_sum += max_ratio
    end
    
    return db_sum / k
end

function calculate_outliers(data, assignments, centers)
    n_samples = size(data, 2)
    distances = zeros(n_samples)
    
    for i in 1:n_samples
        c = assignments[i]
        distances[i] = euclidean(data[:, i], centers[:, c])
    end
    
    threshold = quantile(distances, 0.95)
    outliers = distances .> threshold
    
    return outliers, threshold
end


In [ ]:
if data_pca !== nothing
    results_summary = []
    
    # --- K-Means Analysis ---
    outliers_km, thresh_km = calculate_outliers(data_pca, kmeans_model.assignments, kmeans_model.centers)
    num_outliers_km = sum(outliers_km)
    pct_outliers_km = (num_outliers_km / size(data_pca, 2)) * 100
    
    db_score_km = davies_bouldin(data_pca, kmeans_model.assignments, kmeans_model.centers)
    top_feats_km = get_top_features(names(df_encoded), kmeans_model.centers)
    
    push!(results_summary, Dict(
        "Model" => "K-Means (Optimized)",
        "Optimal_K" => best_k_kmeans,
        "Silhouette" => best_score_kmeans,
        "Davies_Bouldin" => db_score_km,
        "Outliers_Count" => num_outliers_km,
        "Outliers_Pct" => pct_outliers_km,
        "Top_Features" => top_feats_km
    ))
    
    # --- X-Means Analysis ---
    score_xm = calculate_silhouette_score(xmeans_model.assignments, xmeans_model.counts, data_pca)
    db_score_xm = davies_bouldin(data_pca, xmeans_model.assignments, xmeans_model.centers)
    
    outliers_xm, thresh_xm = calculate_outliers(data_pca, xmeans_model.assignments, xmeans_model.centers)
    num_outliers_xm = sum(outliers_xm)
    pct_outliers_xm = (num_outliers_xm / size(data_pca, 2)) * 100
    top_feats_xm = get_top_features(names(df_encoded), xmeans_model.centers)
    
    push!(results_summary, Dict(
        "Model" => "X-Means (Simulated)",
        "Optimal_K" => best_k_xmeans,
        "Silhouette" => score_xm,
        "Davies_Bouldin" => db_score_xm,
        "Outliers_Count" => num_outliers_xm,
        "Outliers_Pct" => pct_outliers_xm,
        "Top_Features" => top_feats_xm
    ))
    
    # --- Autoencoder Analysis ---
    # Note: Latent data is already extracted
    score_ae = calculate_silhouette_score(ae_kmeans.assignments, ae_kmeans.counts, latent_data)
    db_score_ae = davies_bouldin(latent_data, ae_kmeans.assignments, ae_kmeans.centers)
    
    outliers_ae, thresh_ae = calculate_outliers(latent_data, ae_kmeans.assignments, ae_kmeans.centers)
    num_outliers_ae = sum(outliers_ae)
    pct_outliers_ae = (num_outliers_ae / size(data_pca, 2)) * 100
    top_feats_ae = get_top_features(names(df_encoded), ae_kmeans.centers)
    
    push!(results_summary, Dict(
        "Model" => "Autoencoder + K-Means",
        "Optimal_K" => size(ae_kmeans.centers, 2),
        "Silhouette" => score_ae,
        "Davies_Bouldin" => db_score_ae,
        "Outliers_Count" => num_outliers_ae,
        "Outliers_Pct" => pct_outliers_ae,
        "Top_Features" => top_feats_ae
    ))
    
    # Write Summary
    println("\nWriting results to $OUTPUT_FILE...")
    open(OUTPUT_FILE, "w") do f
        write(f, "Advanced Unsupervised Clustering Analysis Summary (Julia)\n")
        write(f, "=======================================================\n\n")
        
        for res in results_summary
            write(f, "Model: $(res["Model"])\n")
            write(f, "-----------------------------------------------\n")
            write(f, "Optimal Clusters (K): $(res["Optimal_K"])\n")
            write(f, "Silhouette Score:     $(round(res["Silhouette"], digits=4))\n")
            write(f, "Davies-Bouldin Index: $(round(res["Davies_Bouldin"], digits=4))\n")
            write(f, "Top 3 Features:       $(join(res["Top_Features"], ", "))\n")
            write(f, "Outliers Detected:    $(res["Outliers_Count"]) ($(round(res["Outliers_Pct"], digits=2))%)\n")
            write(f, "\n")
        end
    end
    
    println("Analysis Complete.")
    println(read(OUTPUT_FILE, String))
end